## Library Imports and Configs

In [1]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import os
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ROOT_PATH = "playground-series-s6e5"

train_df = pd.read_csv(os.path.join(ROOT_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(ROOT_PATH, "test.csv"))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

train_df.shape, test_df.shape # (70 % train, 30 % test)

((439140, 16), (188165, 15))

In [3]:
import itertools
from scipy.stats import chi2_contingency

orig_df = train_df.copy()

categorical_cols = [
    'Driver', 'Compound', 'Race',
    'Year', 'Stint', 'TyreLife_bin',
    'Position_bin'
]

for col1, col2 in itertools.combinations(categorical_cols, 2):

    if col1 in orig_df.columns and col2 in orig_df.columns:

        combined = (
            orig_df[col1].astype(str)
            + ' + ' +
            orig_df[col2].astype(str)
        )

        tab = pd.crosstab(combined, orig_df['PitNextLap'])

        chi2, p, dof, ex = chi2_contingency(tab)

        print(
            f"{col1:12} + {col2:12} | "
            f"chi2={chi2:10.2f} | "
            f"p={p:.6f}"
        )

Driver       + Compound     | chi2=  44602.62 | p=0.000000
Driver       + Race         | chi2=  36077.72 | p=0.000000
Driver       + Year         | chi2=  51118.45 | p=0.000000
Driver       + Stint        | chi2=  75345.00 | p=0.000000
Compound     + Race         | chi2=  62428.47 | p=0.000000
Compound     + Year         | chi2= 103770.53 | p=0.000000
Compound     + Stint        | chi2=  69207.94 | p=0.000000
Race         + Year         | chi2=  77672.74 | p=0.000000
Race         + Stint        | chi2=  87523.75 | p=0.000000
Year         + Stint        | chi2= 140402.25 | p=0.000000


In [4]:
target_counts = train_df['PitNextLap'].value_counts().to_dict()
print("Percentage count of 0.0 PitNextLap")
print(target_counts[0.0] / sum(list(target_counts.values())) * 100)
print("Percentage count of 1.0 PitNextLap")
print(target_counts[1.0] / sum(list(target_counts.values())) * 100)

Percentage count of 0.0 PitNextLap
80.101789862003
Percentage count of 1.0 PitNextLap
19.898210137996994


## Feature Engineering
- Domain Knowledge
- Correlation after feature engineering

In [5]:
def engineer_race_features_old(df):
    """
    Applies feature engineering for race strategy prediction.
    Handles high correlation via ratios and extracts stint-based metrics.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # 1. Handling your high correlation pair (RaceProgress / LapNumber)
    # Adding a small epsilon to avoid division by zero
    df['Progress_Per_Lap_engg'] = df['RaceProgress'] / (df['LapNumber'] + 1e-5)

    # 2. Tyre & Degradation Ratios
    # Captures the intensity of degradation relative to the distance traveled
    df['Deg_Per_Lap_engg'] = df['Cumulative_Degradation'] / (df['LapNumber'] + 1e-5)
    df['Deg_Per_TyreLife_engg'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)

    # 4. Pace Sensitivity (The "Cliff" Detector)
    # How much is the lap time changing relative to tyre age?
    df['Pace_Tyre_Sensitivity_engg'] = df['LapTime_Delta'] / (df['TyreLife'] + 1e-5)

    # 5. Strategic Flags
    # Identify if a driver is losing positions (potential pressure to pit)
    df['Losing_Ground_engg'] = (df['Position_Change'] < 0).astype(int)
    
    # Identify 'Fresh' vs 'Old' tyres based on stint start
    df['Is_Late_Stint_engg'] = (df['TyreLife'] > 20).astype(int) 

    # 6. Interaction Terms
    # Multiplying LapTime_Delta by Cumulative_Degradation to highlight 
    # laps where both pace drops and wear is high
    df['Wear_Pace_Impact_engg'] = df['LapTime_Delta'] * df['Cumulative_Degradation']

    df['Year_Stint_engg'] = (
    df['Year'].astype(str)
    + "_" +
    df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
    df['Compound'].astype(str)
    + "_" +
    df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
    df['Race'].astype(str)
    + "_" +
    df['Stint'].astype(str)
    )

    return df

def engineer_race_features(df, activate=True):

    if not activate:
        return df

    df = df.copy()

    eps = 1e-5

    # =========================================================
    # BASIC PROGRESS FEATURES
    # =========================================================

    df['Progress_Per_Lap_engg'] = (
        df['RaceProgress'] / (df['LapNumber'] + eps)
    )

    df['Remaining_RaceProgress_engg'] = (
        1 - df['RaceProgress']
    )

    df['Remaining_Laps_Ratio_engg'] = (
        (1 - df['RaceProgress']) /
        (df['LapNumber'] + eps)
    )

    # =========================================================
    # DEGRADATION FEATURES
    # =========================================================

    df['Deg_Per_Lap_engg'] = (
        df['Cumulative_Degradation'] /
        (df['LapNumber'] + eps)
    )

    df['Deg_Per_TyreLife_engg'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    df['Deg_x_TyreLife_engg'] = (
        df['Cumulative_Degradation'] *
        df['TyreLife']
    )

    df['Deg_x_Progress_engg'] = (
        df['Cumulative_Degradation'] *
        df['RaceProgress']
    )

    df['Deg_Acceleration_engg'] = (
        df['Cumulative_Degradation'] /
        (df['RaceProgress'] + eps)
    )

    # =========================================================
    # PACE FEATURES
    # =========================================================

    df['Pace_Tyre_Sensitivity_engg'] = (
        df['LapTime_Delta'] /
        (df['TyreLife'] + eps)
    )

    df['Pace_Per_Position_engg'] = (
        df['LapTime_Delta'] /
        (df['Position'] + eps)
    )

    df['LapTime_x_TyreLife_engg'] = (
        df['LapTime_Delta'] *
        df['TyreLife']
    )

    df['LapTime_x_Deg_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['LapTime_x_Progress_engg'] = (
        df['LapTime_Delta'] *
        df['RaceProgress']
    )

    df['Pace_Drop_Flag_engg'] = (
        df['LapTime_Delta'] > 0
    ).astype(int)

    df['Extreme_Pace_Drop_engg'] = (
        df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.90)
    ).astype(int)

    # =========================================================
    # TYRE FEATURES
    # =========================================================

    df['TyreLife_Per_Lap_engg'] = (
        df['TyreLife'] /
        (df['LapNumber'] + eps)
    )

    df['TyreLife_Per_Progress_engg'] = (
        df['TyreLife'] /
        (df['RaceProgress'] + eps)
    )

    df['TyreLife_x_Progress_engg'] = (
        df['TyreLife'] *
        df['RaceProgress']
    )

    df['Fresh_Tyre_Flag_engg'] = (
        df['TyreLife'] <= 5
    ).astype(int)

    df['Medium_Tyre_Flag_engg'] = (
        (df['TyreLife'] > 5) &
        (df['TyreLife'] <= 20)
    ).astype(int)

    df['Old_Tyre_Flag_engg'] = (
        df['TyreLife'] > 20
    ).astype(int)

    # =========================================================
    # POSITION FEATURES
    # =========================================================

    df['Losing_Ground_engg'] = (
        df['Position_Change'] < 0
    ).astype(int)

    df['Gaining_Ground_engg'] = (
        df['Position_Change'] > 0
    ).astype(int)

    df['Position_x_Progress_engg'] = (
        df['Position'] *
        df['RaceProgress']
    )

    df['Position_x_TyreLife_engg'] = (
        df['Position'] *
        df['TyreLife']
    )

    df['Position_Change_Intensity_engg'] = (
        df['Position_Change'] /
        (df['LapNumber'] + eps)
    )

    df['Bad_Position_Flag_engg'] = (
        df['Position'] > 10
    ).astype(int)

    df['Podium_Position_Flag_engg'] = (
        df['Position'] <= 3
    ).astype(int)

    # =========================================================
    # PIT WINDOW FEATURES
    # =========================================================

    df['Potential_Pit_Window_engg'] = (
        (df['TyreLife'] > 15) &
        (df['RaceProgress'] > 0.25)
    ).astype(int)

    df['Late_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] > 0.70)
    ).astype(int)

    df['Early_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] < 0.25)
    ).astype(int)

    # =========================================================
    # INTERACTION FEATURES
    # =========================================================

    df['Wear_Pace_Impact_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['Wear_Position_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position']
    )

    df['Wear_Position_Change_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position_Change']
    )

    df['TyreLife_Position_Interaction_engg'] = (
        df['TyreLife'] *
        df['Position']
    )

    df['TyreLife_Lap_Interaction_engg'] = (
        df['TyreLife'] *
        df['LapNumber']
    )

    df['TyreLife_Stint_Interaction_engg'] = (
        df['TyreLife'] *
        df['Stint']
    )

    df['Lap_Position_Interaction_engg'] = (
        df['LapNumber'] *
        df['Position']
    )

    # =========================================================
    # STINT FEATURES
    # =========================================================

    df['Is_First_Stint_engg'] = (
        df['Stint'] == 1
    ).astype(int)

    df['Is_Second_Stint_engg'] = (
        df['Stint'] == 2
    ).astype(int)

    df['Is_ThirdPlus_Stint_engg'] = (
        df['Stint'] >= 3
    ).astype(int)

    df['Stint_x_Progress_engg'] = (
        df['Stint'] *
        df['RaceProgress']
    )

    df['Stint_x_TyreLife_engg'] = (
        df['Stint'] *
        df['TyreLife']
    )

    # =========================================================
    # CATEGORICAL COMBINATIONS
    # =========================================================

    df['Year_Stint_engg'] = (
        df['Year'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Driver_Compound_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Driver_Race_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Race'].astype(str)
    )

    df['Compound_Stint_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Position_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Position'].astype(str)
    )

    df['Race_Compound_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Race_Year_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Driver_Stint_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    return df


# Apply to your dataframes
train_df_eng = engineer_race_features(train_df)
test_df_eng = engineer_race_features(test_df)

train_df_eng.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Year_Stint_engg,Compound_Year_engg,Race_Stint_engg,Driver_Compound_engg,Driver_Race_engg,Compound_Stint_engg,Compound_Position_engg,Race_Compound_engg,Race_Year_engg,Driver_Stint_engg
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,0.014286,0.285714,0.005714,0.420380,0.538949,819.741,15.013571,29.426188,-0.193949,-0.945499,-294.996,-158.987716,-5.402857,0,0,0.780000,54.599236,27.857143,0,0,1,0,1,5.714286,312.0,0.100000,0,0,1,1,0,-158.987716,168.152,105.095,312.0,1950.0,78.0,400,0,1,0,1.428571,78.0,2022_2,HARD_2022,Canadian Grand Prix_2,D109_HARD,D109_Canadian Grand Prix,HARD_2,HARD_8,Canadian Grand Prix_HARD,Canadian Grand Prix_2022,D109_2
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,0.012821,0.653846,0.024217,-8.266923,-31.886669,-1562.449,-77.263962,-644.801595,-4.659565,-8.154230,-228.319,7280.342719,-11.290500,0,0,0.259259,20.221638,2.423077,0,1,0,1,0,1.384615,28.0,-0.111111,0,0,0,0,0,7280.342719,-892.828,669.621,28.0,189.0,14.0,108,0,1,0,0.692308,14.0,2025_2,HARD_2025,Dutch Grand Prix_2,D086_HARD,D086_Dutch Grand Prix,HARD_2,HARD_4,Dutch Grand Prix_HARD,Dutch Grand Prix_2025,D086_2
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,0.013889,0.180556,0.003060,-1.703881,-4.569498,-2211.638,-82.377931,-122.677961,-0.342727,-0.580000,-165.880,757.988660,-6.178611,0,0,0.372881,26.847130,18.027778,0,0,1,0,1,10.652778,286.0,0.050847,1,0,1,1,0,757.988660,-1306.877,-301.587,286.0,1298.0,66.0,767,0,0,1,2.458333,66.0,2022_3,HARD_2022,Austrian Grand Prix_3,ZON_HARD,ZON_Austrian Grand Prix,HARD_3,HARD_13,Austrian Grand Prix_HARD,Austrian Grand Prix_2022,ZON_3
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0.038461,0.923077,0.461536,-3.661982,-3.661982,-14.648,-0.563385,-95.199624,-3.661982,-1.046284,-14.648,53.640976,-0.563385,0,0,0.999995,25.996620,0.153846,1,0,0,0,0,0.538462,14.0,0.000000,0,0,0,0,1,53.640976,-51.268,-0.000,14.0,4.0,2.0,14,1,0,0,0.076923,2.0,2023_1,MEDIUM_2023,Pre-Season Testing_1,SPE_MEDIUM,SPE_Pre-Season Testing,MEDIUM_1,MEDIUM_7,Pre-Season Testing_MEDIUM,Pre-Season Testing_2023,SPE_1
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,0.013889,0.638889,0.024573,-0.543807,-2.356496,-84.834,-5.105750,-39.153070,1.494164,4.482478,53.790,-126.756135,3.237361,1,0,0.230769,16.614925,2.166667,0,1,0,0,1,0.722222,12.0,0.115385,0,1,0,0,0,-126.756135,-28.278,-42.417,12.0,156.0,18.0,52,0,0,1,1.083333,18.0,2022_3,HARD_2022,Azerbaijan Grand Prix_3,D019_HARD,D019_Azerbaijan Grand Prix,HARD_3,HARD_2,Azerbaijan Grand Prix_HARD,Azerbaijan Grand Prix_2022,D019_3


In [6]:
cat_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(exclude=np.number).columns
num_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(include=np.number).columns
target_col = ['PitNextLap']

## Data Processing
- Encoding - (categorical data) (Label Encoding and OneHotEncoding)
- Normalization - (numerical data)

#### Encoding - Categorical data - Label Encoding and OneHotEncoding

In [7]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# encoding_columns = ['Driver', 'Race', 'Year_Stint_engg', 'Compound_Year_engg', 'Race_Stint_engg']
encoding_columns = ['Driver', 'Race', 'Year_Stint_engg', 'Compound_Year_engg',
       'Race_Stint_engg', 'Driver_Compound_engg', 'Driver_Race_engg',
       'Compound_Stint_engg', 'Compound_Position_engg', 'Race_Compound_engg',
       'Race_Year_engg', 'Driver_Stint_engg']

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

train_df_eng[encoding_columns] = encoder.fit_transform(
    train_df_eng[encoding_columns]
)

test_df_eng[encoding_columns] = encoder.transform(
    test_df_eng[encoding_columns]
)

train_df_eng[[col.replace("engg", "en") if "engg" in col  else col + "_en" for col in encoding_columns]] = encoder.fit_transform(train_df_eng[encoding_columns])
test_df_eng[[col.replace("engg", "en") if "engg" in col  else col + "_en" for col in encoding_columns]] = encoder.transform(test_df_eng[encoding_columns])

train_df_eng_enc = train_df_eng.drop(encoding_columns, axis='columns')
test_df_eng_enc = test_df_eng.drop(encoding_columns, axis='columns')
test_df_eng_enc.head()


,id,Compound,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Driver_en,Race_en,Year_Stint_en,Compound_Year_en,Race_Stint_en,Driver_Compound_en,Driver_Race_en,Compound_Stint_en,Compound_Position_en,Race_Compound_en,Race_Year_en,Driver_Stint_en
0,439140,MEDIUM,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0,0.019231,0.596154,0.028388,-0.237333,-0.237333,-104.664,-2.012769,-12.341028,0.013333,0.070000,5.880,-1.395520,0.113077,1,0,1.000000,51.998712,8.480769,0,0,1,0,0,1.615385,84.0,0.000000,0,0,1,0,0,-1.395520,-19.936,-0.000,84.0,441.0,21.0,84,1,0,0,0.403846,21.0,144.0,6.0,8.0,9.0,36.0,704.0,3747.0,15.0,54.0,25.0,25.0,895.0
1,439141,MEDIUM,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0,0.017241,0.586207,0.024425,-0.082917,-0.082917,-47.760,-0.823448,-4.809050,-0.005375,-0.128999,-3.096,0.256710,-0.053379,0,0,1.000000,57.998598,9.931034,0,0,1,0,0,0.413793,24.0,0.000000,0,1,1,0,0,0.256710,-1.990,-0.000,24.0,576.0,24.0,24,1,0,0,0.413793,24.0,875.0,0.0,8.0,9.0,0.0,2890.0,14631.0,15.0,40.0,2.0,1.0,3777.0
2,439142,MEDIUM,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0,0.019231,0.538462,0.022436,-0.368417,-0.368417,-212.208,-4.080923,-19.157252,0.001708,0.003727,0.984,-0.362522,0.018923,1,0,1.000000,51.998873,11.076923,0,0,1,0,0,5.076923,264.0,0.000000,1,0,1,0,0,-0.362522,-97.262,-0.000,264.0,576.0,24.0,264,1,0,0,0.461538,24.0,295.0,6.0,8.0,9.0,36.0,1276.0,7495.0,15.0,42.0,25.0,25.0,1704.0
3,439143,SOFT,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0,0.012987,0.922078,0.153679,1.374998,2.062495,33.000,0.642857,105.861414,-4.935238,-1.316066,-78.964,-162.863250,-1.538260,0,0,0.666666,51.326746,0.311688,1,0,0,0,1,1.168831,60.0,0.166666,1,0,0,0,1,-162.863250,123.750,8.250,60.0,24.0,8.0,90,0,1,0,0.155844,8.0,137.0,24.0,17.0,14.0,140.0,672.0,3583.0,24.0,66.0,93.0,98.0,856.0
4,439144,HARD,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0,0.013889,0.277778,0.005342,-0.400923,-0.718896,-604.592,-15.056889,-28.866062,0.032069,0.077500,26.970,-19.388640,0.671667,1,0,0.557692,40.153290,20.944444,0,0,1,0,1,8.666667,348.0,0.134615,1,0,1,1,0,-19.388640,-250.176,-145.936,348.0,1508.0,58.0,624,0,1,0,1.444444,58.0,4.0,25.0,17.0,2.0,147.0,20.0,129.0,1.0,3.0,95.0,102.0,28.0


In [8]:
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_data = one_hot_encoder.fit_transform(train_df_eng_enc[['Compound']])
encoded_df = pd.DataFrame(
    encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=train_df_eng_enc.index
)
train_df_eng_enc_v1 = pd.concat([train_df_eng_enc, encoded_df], axis=1).drop('Compound', axis=1)


test_encoded_data = one_hot_encoder.transform(test_df_eng_enc[['Compound']])
test_encoded_df = pd.DataFrame(
    test_encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=test_df_eng_enc.index
)
test_df_eng_enc_v1 = pd.concat([test_df_eng_enc, test_encoded_df], axis=1).drop('Compound', axis=1)

train_df_eng_enc_v1.filter(like='Compound').head()

,Compound_Year_en,Driver_Compound_en,Compound_Stint_en,Compound_Position_en,Race_Compound_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,0.0,654.0,1.0,18.0,27.0,1.0,0.0,0.0,0.0,0.0
1,3.0,545.0,1.0,14.0,34.0,1.0,0.0,0.0,0.0,0.0
2,0.0,2942.0,2.0,4.0,8.0,1.0,0.0,0.0,0.0,0.0
3,9.0,2836.0,15.0,57.0,72.0,0.0,0.0,1.0,0.0,0.0
4,0.0,217.0,2.0,11.0,11.0,1.0,0.0,0.0,0.0,0.0


#### Normalization (MinMax / StandardScaler)

In [9]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

class DataNormalizer:
    def __init__(self, method='standard'):
        """
        Initializes the normalizer with the chosen scaling strategy.
        method: 'minmax' or 'standard'
        """
        self.method = method
        if method == 'minmax':
            self.scaler = MinMaxScaler()
        elif method == 'standard':
            self.scaler = StandardScaler()
        else:
            raise ValueError("Method must be either 'minmax' or 'standard'")

    def fit(self, dataset, num_cols):
        """
        Learns the scaling parameters (mean/std or min/max) from the training set.
        """
        if not all(col in dataset.columns for col in num_cols):
            missing = [c for c in num_cols if c not in dataset.columns]
            raise ValueError(f"Columns missing from dataset: {missing}")
            
        self.scaler.fit(dataset[num_cols])
        print(f"Successfully fitted {self.method} scaler on: {num_cols}")

    def transform(self, dataset, num_cols):
        """
        Applies the learned parameters to scale the dataset.
        """
        df = dataset.copy()
        df[num_cols] = self.scaler.transform(df[num_cols])
        return df

    def fit_transform(self, dataset, num_cols):
        """
        Fits to the data then transforms it. Useful for the initial training set.
        """
        self.fit(dataset, num_cols)
        return self.transform(dataset, num_cols)


normalizer = DataNormalizer(method='standard')

train_df_scaled = normalizer.fit_transform(train_df_eng_enc_v1, num_cols=num_cols)
test_df_scaled = normalizer.transform(test_df_eng_enc_v1, num_cols=num_cols)

train_df_scaled.head()

Successfully fitted standard scaler on: Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'Progress_Per_Lap_engg',
       'Remaining_RaceProgress_engg', 'Remaining_Laps_Ratio_engg',
       'Deg_Per_Lap_engg', 'Deg_Per_TyreLife_engg', 'Deg_x_TyreLife_engg',
       'Deg_x_Progress_engg', 'Deg_Acceleration_engg',
       'Pace_Tyre_Sensitivity_engg', 'Pace_Per_Position_engg',
       'LapTime_x_TyreLife_engg', 'LapTime_x_Deg_engg',
       'LapTime_x_Progress_engg', 'Pace_Drop_Flag_engg',
       'Extreme_Pace_Drop_engg', 'TyreLife_Per_Lap_engg',
       'TyreLife_Per_Progress_engg', 'TyreLife_x_Progress_engg',
       'Fresh_Tyre_Flag_engg', 'Medium_Tyre_Flag_engg', 'Old_Tyre_Flag_engg',
       'Losing_Ground_engg', 'Gaining_Ground_engg', 'Position_x_Progress_engg',
       'Position_x_TyreLife_engg', 'Position_Change_Intensity_engg',
       'Bad_Position_Flag_engg', 

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Driver_en,Race_en,Year_Stint_en,Compound_Year_en,Race_Stint_en,Driver_Compound_en,Driver_Race_en,Compound_Stint_en,Compound_Position_en,Race_Compound_en,Race_Year_en,Driver_Stint_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,0,-1.486487,-0.396946,1.585901,0.221941,2.534531,-0.308849,-0.630046,-0.086333,0.853455,1.487008,1.222548,1.0,-0.091033,-1.487008,-0.509986,0.151346,0.220112,1.196900,1.220405,0.142041,0.024664,-0.023204,-1.253837,-0.019963,-0.979879,-0.660523,-0.333333,-0.019388,-0.019540,2.812363,-0.509979,-1.130327,1.816110,-0.688951,1.320038,0.745101,1.515992,0.098721,-0.899887,-0.433810,1.403299,2.817278,-0.925252,-0.019963,0.747463,0.380384,1.515992,2.875356,2.040728,0.802183,-0.985163,1.545995,-0.519458,0.741509,2.040728,134.0,7.0,1.0,0.0,43.0,654.0,3488.0,1.0,18.0,27.0,28.0,838.0,1.0,0.0,0.0,0.0,0.0
1,1,1.440545,2.519236,0.229628,0.221941,-0.730333,-1.066602,-0.801797,-0.656423,-3.605949,0.033534,-0.774077,0.0,-0.401788,-0.033534,-0.416683,-0.384754,-1.788677,-1.095286,-3.176571,-0.405659,-0.088712,-0.585639,-0.928667,0.365547,-2.250847,-0.660523,-0.333333,-1.234165,-1.149458,-0.509855,-0.509979,0.884700,-0.550627,1.451482,-0.757554,-0.569919,-0.850941,-0.099932,-0.899887,-0.433810,-0.712606,-0.354953,-0.925252,0.365547,-1.124840,2.373826,-0.850941,-0.466697,-0.510165,-0.518896,-0.985163,1.545995,-0.519458,-0.093803,-0.510165,111.0,9.0,23.0,3.0,54.0,545.0,2892.0,1.0,14.0,34.0,39.0,700.0,1.0,0.0,0.0,0.0,0.0
2,2,-1.486487,-0.396946,2.116616,1.274359,0.800072,0.638343,-1.011682,-0.085787,-1.365930,1.902200,0.723392,1.0,-0.175196,-1.902200,-0.523369,0.020257,-0.096360,-1.719948,-3.420249,0.018482,0.020887,0.005312,-0.624164,0.027555,-1.147342,-0.660523,-0.333333,-0.969109,-0.931692,1.528443,-0.509979,-1.130327,1.816110,-0.688951,1.320038,2.245034,1.299301,0.052469,1.111250,-0.433810,1.403299,2.817278,-0.925252,0.027555,-1.855510,-1.055683,1.299301,1.637980,1.562435,2.462581,-0.985163,-0.646833,1.925083,1.909803,1.562435,886.0,2.0,2.0,0.0,14.0,2942.0,14918.0,2.0,4.0,8.0,8.0,3855.0,1.0,0.0,0.0,0.0,0.0
3,3,-0.510810,-0.396946,-1.244581,-0.830476,-1.240468,-0.498287,0.172574,-0.080872,0.335931,-1.029455,-0.025343,0.0,5.036382,1.029455,1.788622,-0.100579,-0.040139,0.394036,0.478171,0.040803,-0.063385,-0.031068,0.113365,-0.008944,0.064819,-0.660523,-0.333333,0.493814,-0.959647,-0.806264,1.960864,-1.130327,-0.550627,-0.688951,-0.757554,-0.826915,-0.967621,0.004622,-0.899887,-0.433810,-0.712606,-0.354953,1.080787,-0.008944,0.360254,0.009275,-0.967621,-0.817793,-0.988458,-0.944175,1.015061,-0.646833,-0.519458,-0.791974,-0.988458,864.0,19.0,8.0,9.0,111.0,2836.0,14364.0,15.0,57.0,72.0,77.0,3703.0,0.0,0.0,1.0,0.0,0.0
4,4,-1.486487,2.519236,0.170660,1.274359,-0.832360,-1.445478,0.856192,0.289790,0.211493,0.092589,0.723392,0.0,-0.175196,-0.

## WoE Transformation
- WoE
- IV

In [11]:
# %pip install --upgrade optbinning
# %pip install scikit-learn==1.5.2

In [12]:
import time
import numpy as np
import pandas as pd
from optbinning import OptimalBinning
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

MIN_BIN_BUREAU = 0.03
MIN_BIN_APP    = 0.05

TARGET = target_col[0]
train_pd = train_df_scaled
test_pd = test_df_scaled
final_vars = train_df_scaled.columns

# -------------------------------------------------------
# Containers
# -------------------------------------------------------

binning_objects = {}
woe_errors = []
iv_summary = {}


# Create transformed datasets
woe_train = pd.DataFrame(index=train_pd.index)
woe_test  = pd.DataFrame(index=test_pd.index)

# Add target column
woe_train[TARGET] = train_pd[TARGET]
y_train = train_pd[TARGET].values

print(f"Fitting WoE binning on {len(train_pd):,} rows for {len(final_vars):,} features...")

t0 = time.time()

# -------------------------------------------------------
# Fit + Transform
# -------------------------------------------------------

for i, feat in enumerate(final_vars):

    try:
        # -------------------------------
        # Feature data
        # -------------------------------
        x_train = train_pd[feat].values
        x_test  = test_pd[feat].values

        # Detect categorical
        is_cat = (
            train_pd[feat].dtype == "object"
            or str(train_pd[feat].dtype) == "category"
        )

        dtype = "categorical" if is_cat else "numerical"

        # Min bin size
        min_bin = (
           MIN_BIN_BUREAU
        )

        # -------------------------------
        # Initialize Optimal Binning
        # -------------------------------
        ob = OptimalBinning(
            name=feat,
            dtype=dtype,
            min_bin_size=min_bin,
            max_n_bins=10,
            solver="cp",
            monotonic_trend="auto"
        )

        # -------------------------------
        # Fit on TRAIN only
        # -------------------------------
        ob.fit(x_train, y_train)

        # Save object
        binning_objects[feat] = ob

        # -------------------------------
        # WoE Transformation
        # -------------------------------
        woe_train[feat] = ob.transform(
            x_train,
            metric="woe"
        )

        woe_test[feat] = ob.transform(
            x_test,
            metric="woe"
        )

        # -------------------------------
        # Information Value
        # -------------------------------
        bt = ob.binning_table.build()

        iv_summary[feat] = bt.loc["Totals", "IV"]

    except Exception as e:

        woe_errors.append({
            "feature": feat,
            "error": str(e)
        })

    # Progress logging
    if (i + 1) % 100 == 0 or (i + 1) == len(final_vars):

        elapsed = time.time() - t0

        print(
            f"{i+1:>4}/{len(final_vars)} "
            f"| {elapsed:.0f}s "
            f"| errors: {len(woe_errors)}"
        )

# -------------------------------------------------------
# IV Summary
# -------------------------------------------------------

iv_df = (
    pd.DataFrame.from_dict(
        iv_summary,
        orient="index",
        columns=["IV"]
    )
    .sort_values("IV", ascending=False)
)

# -------------------------------------------------------
# Final Logs
# -------------------------------------------------------

print("\nBinning complete")
print(f"Fitted features : {len(binning_objects):,}")
print(f"Errors          : {len(woe_errors):,}")

if len(woe_errors) > 0:
    print("\nSample Errors:")
    print(pd.DataFrame(woe_errors).head())

print("\nTop IV Features:")
print(iv_df.head(20))

# -------------------------------------------------------
# Final Training Data
# -------------------------------------------------------

train_df_woe = woe_train
test_df_woe = woe_test

X_woe, y_woe = train_df_woe.drop(['id', 'PitNextLap'], axis='columns'), train_df_woe['PitNextLap'].astype(int)
X_train_woe, X_test_woe, y_train_woe, y_test_woe = train_test_split(X_woe, y_woe, stratify=y, test_size=0.3, random_state=42)


count_neg = (y_woe == 0).sum()
count_pos = (y_woe == 1).sum()
scale_weight = count_neg / count_pos
print(f"Scale Weight: {scale_weight:.2f}")


xgb_model = XGBClassifier(
    n_estimators      = 2000,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    scale_pos_weight = scale_weight
)

xgb_model.fit(X_train_woe, y_train_woe)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test_woe)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train_woe)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test_woe, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train_woe, y_prob_train_xgb):.4f}")

(CVXPY) May 24 12:14:44 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) May 24 12:14:44 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


Fitting WoE binning on 439,140 rows for 73 features...
  73/73 | 36s | errors: 1

Binning complete
Fitted features : 72
Errors          : 1

Sample Errors:
      feature         error
0  PitNextLap  'PitNextLap'

Top IV Features:
                                       IV
Year_Stint_en                    1.781905
Year                             1.369349
TyreLife_Per_Lap_engg            1.145130
Progress_Per_Lap_engg            1.066579
LapTime_x_Progress_engg          0.992482
Stint_x_TyreLife_engg            0.989035
TyreLife_Stint_Interaction_engg  0.989035
LapTime_x_TyreLife_engg          0.987595
Stint                            0.978355
Compound_Stint_en                0.913761
Stint_x_Progress_engg            0.888792
Is_First_Stint_engg              0.887936
Compound_Year_en                 0.858427
LapTime_Delta                    0.701138
TyreLife_Lap_Interaction_engg    0.696578
Remaining_Laps_Ratio_engg        0.676586
RaceProgress                     0.627753
Remaining_Race

NameError: name 'y' is not defined

## Choosing Best Model 

In [14]:
def submission(model, test_data, file_name:str):
    pred_data = test_data
    if 'id' in test_data.columns:
        pred_data = test_data.drop(['id'], axis='columns')
        print("!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!")
        
    predictions = model.predict_proba(pred_data)[:, 1]
    sub_df['PitNextLap'] = predictions
    sub_df[['id', 'PitNextLap']].to_csv(file_name, index=False)
    print(f"Submissions saved to {file_name} path!!!!!!!!!!!!!!")

## Target Encoding - OOF

In [15]:
from sklearn.model_selection import KFold

encoded_cat_cols = [col for col in train_df_scaled.columns if '_en' in col.lower() and '_engg' not in col.lower()]
encoded_cat_cols

train_df_scaled_v1 = train_df_scaled.copy()
test_df_scaled_v1 = test_df_scaled.copy()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in encoded_cat_cols:
    train_df_scaled_v1[f'{col}_mean'] = 0.0
    train_df_scaled_v1[f'{col}_std'] = 0.0

for train_idx, val_idx in kf.split(train_df_scaled_v1):
    train_fold = train_df_scaled_v1.iloc[train_idx]
    val_fold = train_df_scaled_v1.iloc[val_idx]

    for col in encoded_cat_cols:
        stats = train_fold.groupby(col)[target_col[0]].agg(['mean', 'std'])
        val_fold = val_fold.merge(stats, on=col, how='left')

        train_df_scaled_v1.loc[val_idx, f'{col}_mean'] = val_fold['mean'].values
        train_df_scaled_v1.loc[val_idx, f'{col}_std'] = val_fold['std'].values

        val_fold.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    train_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = train_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

for col in encoded_cat_cols:
    stats = train_df_scaled_v1.groupby(col)[target_col[0]].agg(['mean', 'std'])
    test_df_scaled_v1 = test_df_scaled_v1.merge(stats, on=col, how='left')

    test_df_scaled_v1[f'{col}_mean'] = test_df_scaled_v1['mean']
    test_df_scaled_v1[f'{col}_std'] = test_df_scaled_v1['std']

    test_df_scaled_v1.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    test_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = test_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)


X, y = train_df_scaled_v1.drop(['id', 'PitNextLap'], axis='columns'), train_df_scaled_v1['PitNextLap'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

## Searching for best hyperparameters
- optuna

In [17]:
!pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.5 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 1.3 MB/s eta 0:00:02
   ------------------------ --------------- 1.3/2.1 MB 1.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.6 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.6 MB/s  0:00:01

   ------ --------------------------------- 1/6 [greenlet]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## XGBoost Optuna search

In [ ]:
import optuna
import xgboost as xgb

# DMatrix --------------------(XGBoost native format - faster than numpy arrays)--------------------------------
dtrain_xgb = xgb.DMatrix(X_train_woe, label=y_train_woe, feature_names=list(X_train_woe.columns))
dval_xgb = xgb.DMatrix(X_test_woe, label=y_test_woe,  feature_names=list(X_test_woe.columns))

count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

print("Running Optuna search (50 trials)...")
print("Optimising Primary OOT AUC directly\n")

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "verbosity": 0,
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": scale_weight,
        "seed": 42,
        "nthread": -1,
    }

    er = {}
    m = xgb.train(
        params,
        dtrain_xgb,
        num_boost_round=2000,
        evals=[(dval_xgb, "val")],
        early_stopping_rounds=30,
        evals_result=er,
        verbose_eval=False,
    )

    return m.best_score  # best OOT AUC

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_val_auc = study.best_value
best_val_gini = 2 * best_val_auc - 1

print(f"\nOptuna complete:")
print(f"  Best OOT AUC  : {best_val_auc:.5f}")
print(f"  Best OOT Gini : {best_val_gini:.5f}")
print(f"  Best params   : {study.best_params}")

# --- Refit final model with best params ---
final_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "verbosity": 0,
    "scale_pos_weight": scale_weight,
    "seed": 42,
    "nthread": -1,
    **study.best_params,
}

best_params_used = final_params

print(f"\nFinal XGBoost model ready")

[I 2026-05-16 17:06:26,161] A new study created in memory with name: no-name-190eed95-8255-46ee-9909-001fb9339ffc


Running Optuna search (50 trials)...
Optimising Primary OOT AUC directly



  0%|          | 0/50 [00:00<?, ?it/s]

In [17]:
count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

best_xgb_params = {'max_depth': 8, 'learning_rate': 0.04101781341683126, 'min_child_weight': 80, 'subsample': 0.9440420600689602, 'colsample_bytree': 0.5538332346301195, 'colsample_bylevel': 0.8884695922566868, 'reg_alpha': 0.024826827681311267, 'reg_lambda': 0.01362810712606467, 'gamma': 0.9474271992777541}

xgb_model = XGBClassifier(
    **best_xgb_params,
    n_estimators      = 2000,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    scale_pos_weight = scale_weight
)

xgb_model.fit(X_train.values, y_train.values)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")

ROC AUC Score - XGBoost
Testing:  0.9510
Training: 0.9771


In [18]:
xgb_model_final_v1 = XGBClassifier(
    **best_xgb_params,
    n_estimators      = 2000,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    scale_pos_weight = scale_weight
)

xgb_model_final_v1.fit(X.values, y.values)
submission(xgb_model_final_v1, test_df_scaled_v1, file_name="First_XGB_solution_23_05_2026_New_FE.csv")

!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to First_XGB_solution_23_05_2026_New_FE.csv path!!!!!!!!!!!!!!


## LightGBM Optuna search

In [21]:
# import lightgbm as lgb

# # DMatrix --------------------(LightGBM native format - faster than numpy arrays)--------------------------------
# dtrain_lgb = lgb.Dataset(X_train, label=y_train, feature_name=list(X_train.columns))
# dval_lgb = lgb.Dataset(X_test, label=y_test,  feature_name=list(X_test.columns))

# print("Running Optuna search (50 trials)...")

# def objective(trial):
#     params = {
#         "objective"         : "binary",
#         "metric"            : "auc",
#         "verbosity"         : -1,
#         "boosting_type"     : "gbdt",
#         "feature_pre_filter": False,  # <-- ADD THIS LINE TO FIX THE ERROR
#         "num_leaves"        : trial.suggest_int("num_leaves", 31, 255),
#         "max_depth"         : trial.suggest_int("max_depth", 4, 8),
#         "learning_rate"     : trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
#         "min_child_samples" : trial.suggest_int("min_child_samples", 20, 100),
#         "subsample"         : trial.suggest_float("subsample", 0.6, 1.0),
#         "subsample_freq"    : 1,
#         "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
#         "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
#         "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
#         "scale_pos_weight"  : scale_weight,
#         "n_estimators"      : 2000,
#         "random_state"      : 42,
#         "n_jobs"            : -1,
#     }

#     cb = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
#     m = lgb.train(params, dtrain_lgb,
#                   valid_sets=[dval_lgb], valid_names=["oot_primary"],
#                   callbacks=cb)

#     prob = m.predict(X_test.values)
#     return roc_auc_score(y_test, prob)

# study_lgb = optuna.create_study(direction="maximize",
#                                  sampler=optuna.samplers.TPESampler(seed=42))
# study_lgb.optimize(objective, n_trials=50,
#                    show_progress_bar=True)

# print(f"\nBest OOT AUC : {study_lgb.best_value:.5f}")
# print(f"Best OOT Gini : {2*study_lgb.best_value-1:.5f}")
# print(f"Best params   : {study_lgb.best_params}")

# # Save best params
# best_params_lgb = {
#     "objective"        : "binary",
#     "metric"           : "auc",
#     "verbosity"        : -1,
#     "boosting_type"    : "gbdt",
#     "scale_pos_weight" : scale_weight,
#     "n_estimators"     : 2000,
#     "random_state"     : 42,
#     "n_jobs"           : -1,
#     **study_lgb.best_params
# }
# best_params_lgb["subsample_freq"] = 1

# print(f"\n/ Final model selected")

In [ ]:
# lgb_model = LGBMClassifier(
#     n_estimators      = 1500,
#     learning_rate     = 0.02,
#     num_leaves        = 255,
#     min_child_samples = 20,
#     subsample         = 0.8,
#     subsample_freq    = 1,
#     colsample_bytree  = 0.8,
#     reg_alpha         = 0.1,
#     reg_lambda        = 0.1,
#     random_state      = 42,
#     n_jobs            = -1
# )

lgb_best_params = {'num_leaves': 153, 'max_depth': 8, 'learning_rate': 0.022283855802695055, 'min_child_samples': 93, 'subsample': 0.9779576595270691, 'colsample_bytree': 0.5726559097383321, 'reg_alpha': 4.979689286697664, 'reg_lambda': 0.00921590641601282}
# lgb_best_params = {'learning_rate': 0.04908509811664363, 'n_estimators': 480, 'max_depth': 12, 'num_leaves': 254, 'min_child_samples': 144, 'subsample': 0.5594219974551418, 'colsample_bytree': 0.5684090737873079, 'reg_alpha': 0.33033595886058253, 'reg_lambda': 0.6808813914868261, 'min_split_gain': 0.08550184914506787}
best_params_lgb = {
    "objective"        : "binary",
    "metric"           : "auc",
    "verbosity"        : -1,
    "boosting_type"    : "gbdt",
    "scale_pos_weight" : scale_weight,
    "n_estimators"     : 2000,
    "random_state"     : 42,
    "n_jobs"           : -1,
    **lgb_best_params
}

lgb_model = LGBMClassifier(
   **best_params_lgb
)

lgb_model.fit(X_train.values, y_train.values)

# --- LightGBM Evaluation ---
y_prob_test_lgb = lgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_lgb = lgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - LightGBM")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")

ROC AUC Score - LightGBM
Testing:  0.9508
Training: 0.9846


In [20]:
lgb_model_final_v1 = LGBMClassifier(
   **best_params_lgb
)

lgb_model_final_v1.fit(X.values, y.values)
submission(lgb_model_final_v1, test_df_scaled_v1, file_name="First_LGB_solution_23_05_2026_LGB_New_Features.csv")

!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to First_LGB_solution_23_05_2026_LGB_New_Features.csv path!!!!!!!!!!!!!!


## CatBoost Optuna search

In [ ]:
# import catboost as cb
# from catboost import Pool
# from sklearn.metrics import roc_auc_score
# import optuna

# # Assuming dtrain_cb and dval_cb are already created as cb.Pool objects
# dtrain_cb = Pool(X_train, y_train)
# dval_cb = Pool(X_test, y_test)

# print("Running Optuna search for CatBoost (50 trials)...")

# def objective(trial):
#     params = {
#         "loss_function": "Logloss",
#         "eval_metric": "AUC",
#         "verbose": False,
#         "task_type": "CPU",  # Change to "GPU" if you have a GPU available
#         "iterations": 2000,   # Equivalent to n_estimators

#         # Hyperparameters to tune
#         "depth": trial.suggest_int("depth", 4, 8),                       # Max depth of trees
#         "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-4, 10.0, log=True),  # L2 regularization
#         "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness for scoring splits
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0), # Controls bootstrap weights (Bayesian bootstrap)
#         "border_count": trial.suggest_int("border_count", 32, 255),       # Number of splits for numerical features

#         # Scale position weight handling
#         "scale_pos_weight": scale_weight,
#         "random_seed": 42,
#         "thread_count": -1,
#     }

#     # Initialize the model
#     model = cb.CatBoostClassifier(**params)

#     # Train the model with early stopping
#     model.fit(
#         dtrain_cb,
#         eval_set=dval_cb,
#         early_stopping_rounds=50,
#         verbose=False
#     )

#     # Predict probabilities (CatBoost predict_proba returns [prob_0, prob_1])
#     # Pass the Pool object or X_test. pool format is preferred.
#     prob = model.predict_proba(dval_cb)[:, 1]

#     return roc_auc_score(y_test, prob)

# # Create and run the study
# study_cb = optuna.create_study(direction="maximize",
#                                 sampler=optuna.samplers.TPESampler(seed=42))
# study_cb.optimize(objective, n_trials=50, show_progress_bar=True)

# print(f"\nBest OOT AUC : {study_cb.best_value:.5f}")
# print(f"Best OOT Gini : {2*study_cb.best_value-1:.5f}")
# print(f"Best params   : {study_cb.best_params}")

# # Save best params
# best_params_cb = {
#     "loss_function": "Logloss",
#     "eval_metric": "AUC",
#     "verbose": False,
#     "iterations": 2000,
#     "scale_pos_weight": scale_weight,
#     "random_seed": 42,
#     "thread_count": -1,
#     **study_cb.best_params
# }

# print(f"\n/ Final CatBoost model selected")

[I 2026-05-16 15:40:59,603] A new study created in memory with name: no-name-bc729d11-1617-4751-b083-cb10bb1158f4


Running Optuna search for CatBoost (50 trials)...


Best trial: 0. Best value: 0.949014:   2%|▏         | 1/50 [02:06<1:42:54, 126.01s/it]

[I 2026-05-16 15:43:05,610] Trial 0 finished with value: 0.9490136707666309 and parameters: {'depth': 5, 'learning_rate': 0.09237421878009823, 'l2_leaf_reg': 0.4570563099801455, 'random_strength': 0.24810409748678125, 'bagging_temperature': 0.15601864044243652, 'border_count': 66}. Best is trial 0 with value: 0.9490136707666309.


Best trial: 0. Best value: 0.949014:   4%|▍         | 2/50 [03:57<1:33:45, 117.20s/it]

...

Best trial: 0. Best value: 0.949014:   8%|▊         | 4/50 [07:43<1:32:06, 120.13s/it]

[I 2026-05-16 15:48:43,596] Trial 3 finished with value: 0.9482569745722936 and parameters: {'depth': 6, 'learning_rate': 0.03195879743482072, 'l2_leaf_reg': 0.11462107403425033, 'random_strength': 0.003613894271216527, 'bagging_temperature': 0.29214464853521815, 'border_count': 114}. Best is trial 0 with value: 0.9490136707666309.


Best trial: 0. Best value: 0.949014:   8%|▊         | 4/50 [08:14<1:34:51, 123.74s/it]

[W 2026-05-16 15:49:14,532] Trial 4 failed with parameters: {'depth': 6, 'learning_rate': 0.07076922519051031, 'l2_leaf_reg': 0.0009962513222055108, 'random_strength': 0.11400863701127326, 'bagging_temperature': 0.5924145688620425, 'border_count': 42} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\admin\AppData\Local\Temp\ipykernel_3952\437359255.py", line 38, in objective
    model.fit(
    ~~~~~~~~~^
        dtrain_cb,
        ^^^^^^^^^^
    ...<2 lines>...
        verbose=False
        ^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\catboost\core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_mod

KeyboardInterrupt: 

In [53]:
model_names = ['xgb', 'lgb']
model_scores = [roc_auc_score(y_test, y_prob_test_xgb), roc_auc_score(y_test, y_prob_test_lgb)]
# model_scores = [0.95032, 0.9502]
weights = [score / sum(model_scores) for score in model_scores]
model_weights = dict(zip(model_names, weights))

lgb_predictions = pd.read_csv("First_LGB_solution_16_05_2026_optuna_v1.csv")
xgb_predictions = pd.read_csv("First_XGB_solution_16_05_2026_optuna_v1.csv")

final_prediction = model_weights['xgb'] * xgb_predictions['PitNextLap'] + model_weights['lgb'] * lgb_predictions['PitNextLap']
final_predictions_df = pd.DataFrame({'id': test_df_scaled_v1['id'], 'PitNextLap':final_prediction})
final_predictions_df.head()

,id,PitNextLap
0,439140,0.012101
1,439141,0.013177
2,439142,0.011122
3,439143,0.404891
4,439144,0.947877


In [54]:
final_predictions_df.to_csv("Blending_LGB_XGB_solution_16_05_2026_optuna_v1.csv", index=False)

In [22]:
cat_model_oot = CatBoostClassifier(
    verbose=0,
    auto_class_weights='Balanced'
)

cat_model_oot.fit(X_train, y_train)

y_prob_test_cat = cat_model_oot.predict_proba(X_test)[:, 1]
y_prob_train_cat = cat_model_oot.predict_proba(X_train)[:, 1]

print("ROC AUC Score - CatBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_cat):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_cat):.4f}")

ROC AUC Score - CatBoost
Testing:  0.9500
Training: 0.9630


In [24]:
cat_model_final_v1 = CatBoostClassifier(
    eval_metric = "AUC",
    verbose=0,
    auto_class_weights='Balanced'
)

cat_model_final_v1.fit(X, y)
submission(cat_model_final_v1, test_df_scaled_v1, file_name="CAT_Solution_23_05_2026_New_Features.csv")

!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to CAT_Solution_23_05_2026_New_Features.csv path!!!!!!!!!!!!!!


In [49]:
model_names = ['xgb', 'lgb', 'cat']
model_scores = [roc_auc_score(y_test, y_prob_test_xgb), roc_auc_score(y_test, y_prob_test_lgb), roc_auc_score(y_test, y_prob_test_cat)]
weights = [score / sum(model_scores) for score in model_scores]
model_weights = dict(zip(model_names, weights))

lgb_predictions = pd.read_csv("First_LGB_solution_11_05_2026_v1.csv")
xgb_predictions = pd.read_csv("First_XGB_solution_11_05_2026_v1.csv")
cat_predictions = pd.read_csv("CatBoost_Solution_11_05_2026.csv")

final_predictions_v1 = model_weights['xgb'] * xgb_predictions['PitNextLap'] + model_weights['lgb'] * lgb_predictions['PitNextLap'] + model_weights['cat'] * cat_predictions['PitNextLap']
final_predictions_v1_df = pd.DataFrame({'id': test_df_scaled_v1['id'], 'PitNextLap':final_predictions_v1})
final_predictions_v1_df.head()

,id,PitNextLap
0,439140,0.012299
1,439141,0.012231
2,439142,0.011135
3,439143,0.285833
4,439144,0.929646


In [51]:
final_predictions_v1_df.to_csv("Blending_XGB_LGB_CAT_11_05_2026_v1.csv", index=False)